# 07 — Gradio Web Demo

Launch an interactive web interface for DR detection.
Upload any retinal fundus image and get an instant prediction.

In [ ]:
import os, sys, importlib, shutil

# --- CONFIGURE DATA SOURCE ---
# "kaggle" = download from Kaggle (~1-3 min, recommended for Colab)
# "drive"  = copy from Google Drive (~15-30 min, fallback)
# "local"  = data already at expected path (no copy needed)
DATA_SOURCE = "kaggle"

IS_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/diabetic_retinopathy'
    %pip install -q timm albumentations kaggle gradio

    LOCAL_DATA = '/content/dataset'
    if not os.path.isdir(os.path.join(LOCAL_DATA, 'resized_train_cropped')):
        if DATA_SOURCE == "kaggle":
            print('Upload your kaggle.json when prompted...')
            from google.colab import files
            uploaded = files.upload()  # user uploads kaggle.json
            os.makedirs('/root/.kaggle', exist_ok=True)
            shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
            os.chmod('/root/.kaggle/kaggle.json', 0o600)
            !kaggle datasets download -d tanlikesmath/diabetic-retinopathy-resized -p /content/ -q
            os.makedirs(LOCAL_DATA, exist_ok=True)
            import zipfile
            with zipfile.ZipFile('/content/diabetic-retinopathy-resized.zip', 'r') as z:
                z.extractall(LOCAL_DATA)
            os.remove('/content/diabetic-retinopathy-resized.zip')
            print('Kaggle download complete!')
        elif DATA_SOURCE == "drive":
            print('Copying dataset from Drive to local SSD (15-30 min)...')
            shutil.copytree(
                os.path.join(PROJECT_ROOT, 'dataset'), LOCAL_DATA,
                dirs_exist_ok=True,
            )
            print('Drive copy complete!')
        else:
            print(f'DATA_SOURCE="{DATA_SOURCE}" — assuming data is already in place.')
            if not os.path.isdir(LOCAL_DATA):
                raise FileNotFoundError(
                    f"Local data not found at {LOCAL_DATA}. "
                    f"Set DATA_SOURCE to 'kaggle' or 'drive'."
                )
    else:
        print('Dataset already on local SSD.')
else:
    PROJECT_ROOT = os.path.abspath('..')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.config
importlib.reload(src.config)
from src import config

config.ensure_dirs()

print(f'Images dir: {config.IMAGES_DIR}')
print(f'  exists: {os.path.isdir(config.IMAGES_DIR)}')
print(f'Outputs dir: {config.OUTPUTS_DIR}')

In [ ]:
from src.app import create_app

# Launch the Gradio demo
# On Colab, this creates a public URL you can share
app = create_app()
app.launch(share=True, debug=False)